# l-C2ST-NF: Local Classifier Two-Sample Test

Tests whether the learned posterior $q_\phi(\theta|x)$ matches the true posterior $p(\theta|x)$.

**Method:** Train a classifier in the normalizing flow's latent space $Z$ to distinguish:
- $z \sim \mathcal{N}(0, I)$ (base distribution)
- $z = T^{-1}(\theta; x)$ (inverse-transformed ground truth $\theta$)

If well-calibrated → classifier accuracy ≈ 0.5 → p-value > α → do not reject $H_0$.

**Reference:** Linhart et al. (2023)

In [ ]:
import sys
from pathlib import Path
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import dill as pickle

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))

from sbi4atmret.config.configs import BaseConfig
from sbi4atmret.models.ModelBase import BaseModel
from sbi4atmret.utils.checkpoint import load_checkpoint, load_model_state
from sbi4atmret.datasets.DatasetBase import Dataset
from sbi4atmret.runtime.batch_processor import BatchProcessor
from sbi4atmret.evaluation.setup_evaluation import setup_evaluation
from sbi4atmret.evaluation.lc2st import LC2STEvaluator, LC2STResult

## 1. Setup

In [ ]:
config_path = project_root / "experiments/config_MiriGeminiHST_cloudfree.yaml"
checkpoint_path = Path("path/to/checkpoints/latest.pt")

with open(config_path) as f:
    config = BaseConfig(**yaml.safe_load(f))

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build model
model = BaseModel(config).build()
cp = load_checkpoint(checkpoint_path, device)
load_model_state(model.estimator, cp)
model.estimator.to(device).eval()

# Build dataset and evaluation context
dataset = Dataset(config)
ctx = setup_evaluation(config=config, dataset=dataset,
                       checkpoint_path=checkpoint_path, device=device)

print(f"Model loaded, device: {device}")

In [ ]:
# Build the LC2ST evaluator with shared state
lc2st_eval = LC2STEvaluator.__new__(LC2STEvaluator)
lc2st_eval.__dict__.update({
    'net': model.estimator,
    'pipe': ctx.runtime.domain.pipe,
    'noise': ctx.runtime.domain.noise,
    'batch_processor': BatchProcessor(
        pipe=ctx.runtime.domain.pipe,
        noise=ctx.runtime.domain.noise,
        device=device,
    ),
    'test_keys': ctx.test_lists[0],
    'test_loaders': ctx.test_lists[1],
    'device': device,
    'config': config,
    'observation': ctx.runtime.domain.observation,
    'x_obs': torch.from_numpy(
        ctx.runtime.domain.observation.full_observation
    ).unsqueeze(0).float().to(device),
})

# Save paths
save_path = Path("lc2st_results")
save_path.mkdir(exist_ok=True)

## 2. Collect Calibration Data

In [ ]:
# Collect (theta, x, posterior_samples) from the test set
# This runs the batch processor + posterior sampling
theta_cal, x_cal, post_cal = lc2st_eval.collect_calibration_data(n_batches=128)

print(f"Calibration data: theta={theta_cal.shape}, x={x_cal.shape}, post={post_cal.shape}")

# Save for reuse
torch.save({
    'theta': theta_cal,
    'x': x_cal,
    'post_samples': post_cal,
}, save_path / 'calibration_data.pt')

## 3. Train l-C2ST-NF

In [ ]:
# Or load from disk:
# data = torch.load(save_path / 'calibration_data.pt')
# theta_cal, x_cal, post_cal = data['theta'], data['x'], data['post_samples']

n_params = theta_cal.shape[-1]

# Classifier hyperparameters
clf_kwargs = {
    'hidden_layer_sizes': (10 * n_params, 10 * n_params),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,
    'batch_size': 2048,
    'learning_rate': 'adaptive',
    'learning_rate_init': 1e-3,
    'max_iter': 1000,
    'tol': 1e-4,
    'n_iter_no_change': 50,
}

result = lc2st_eval.run_lc2st(
    theta=theta_cal,
    x=x_cal,
    post_samples=post_cal,
    N=50000,
    num_ensemble=10,
    num_trials_null=50,
    num_folds=3,
    classifier='mlp',
    clf_kwargs=clf_kwargs,
    save_path=save_path,
)

print("l-C2ST training complete. Model saved.")

## 4. Evaluate on Real Observation

In [ ]:
# Load trained l-C2ST (if resuming)
# with open(save_path / 'lc2st_nf.pkl', 'rb') as f:
#     lc2st_nf = pickle.load(f)

lc2st_nf = result.lc2st_nf

# Prepare observations to test
# x_obs is the real observation; optionally add simulated test cases
x_obs_cpu = lc2st_eval.x_obs.cpu()

obs_labels = ["Real observation"]
x_test = x_obs_cpu  # (1, D_obs)

# Evaluate
p_values, rejections, fig_t, fig_pp = lc2st_eval.evaluate_observations(
    lc2st_nf,
    x_observations=x_test,
    obs_labels=obs_labels,
    conf_alpha=0.05,
)

print(f"\nResults:")
for label, pv, rej in zip(obs_labels, p_values, rejections):
    print(f"  {label}: p-value={pv:.4f}, reject H0={rej}")

In [ ]:
# Save T-distribution plot
fig_t.savefig(save_path / "lc2st_Tdist.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Save PP-plot
fig_pp.savefig(save_path / "lc2st_pp_plot.pdf", bbox_inches="tight")
plt.show()

## 5. Pair-grid with Probability Intensity

Visualize which regions of parameter space the classifier flags as mismatched.

In [ ]:
from sbi.diagnostics.graphical_diagnostics import (
    compute_dfs_with_probas_marginals,
    eval_space_with_proba_intensity,
)

# Sample from base distribution and transform to parameter space
n_params = theta_cal.shape[-1]
flow_base_dist = torch.distributions.MultivariateNormal(
    torch.zeros(n_params), torch.eye(n_params)
)

samples_z = flow_base_dist.sample(torch.Size([10000])).to(device)

# Forward transform: z → θ
with torch.no_grad():
    posterior = lc2st_eval.net.flow_forward(x_test[:1].to(device))
    samples_theta = samples_z.clone()
    for trns in posterior.transforms:
        samples_theta = trns(samples_theta)

samples_theta = samples_theta.cpu()

# Get classifier probabilities
probas, _ = lc2st_nf.get_scores(
    theta_o=samples_z.cpu(),
    x_o=x_test[0].to(device),
    trained_clfs=lc2st_nf.trained_clfs,
    return_probs=True,
)

print(f"Samples: {samples_theta.shape}, Probas: {probas[0].shape}")

In [ ]:
import matplotlib.cm as cm

# Select a subset of parameters to plot (e.g., first 6)
params_to_plot = list(range(min(6, n_params)))
n_plot = len(params_to_plot)

dfs = compute_dfs_with_probas_marginals(probas, P_eval=samples_theta)

cmap = cm.get_cmap("Spectral_r")

fig, axs = plt.subplots(n_plot, n_plot, figsize=(12, 12),
                         sharex=False, sharey=False)

param_names = [p.name for p in config.prior_config.parameters]

for row, i in enumerate(params_to_plot):
    for col, j in enumerate(params_to_plot):
        ax = axs[row][col]
        if row == col:
            # Diagonal: 1D marginal
            eval_space_with_proba_intensity(
                df_probas=dfs[f"{i}"],
                dim=1, z_space=False, n_bins=20,
                vmin=0.2, vmax=0.8, cmap=cmap,
                show_colorbar=False, ax=ax,
            )
        elif row > col:
            # Lower triangle: 2D
            key = f"{j}_{i}"
            if key in dfs:
                eval_space_with_proba_intensity(
                    df_probas=dfs[key],
                    dim=2, z_space=False, n_bins=20,
                    vmin=0.2, vmax=0.8, cmap=cmap,
                    show_colorbar=False, ax=ax,
                )
        else:
            ax.set_visible(False)

        if row == n_plot - 1:
            ax.set_xlabel(param_names[j], fontsize=8)
        if col == 0 and row != 0:
            ax.set_ylabel(param_names[i], fontsize=8)

fig.suptitle(r"l-C2ST-NF: Predicted probability $\ell(z)$ in $\Theta$-space",
             fontsize=12)
plt.tight_layout()
fig.savefig(save_path / "lc2st_pairgrid.pdf", bbox_inches="tight")
plt.show()

## 6. Summary

In [ ]:
print("=" * 50)
print("l-C2ST-NF Summary")
print("=" * 50)
print(f"Calibration samples: {len(theta_cal)}")
print(f"Samples used for test: {min(50000, len(theta_cal))}")
print(f"Classifier: MLP with {clf_kwargs['hidden_layer_sizes']}")
print(f"Ensemble size: 10")
print(f"Null trials: 50")
print()
print("Results on observations:")
for label, pv, rej in zip(obs_labels, p_values, rejections):
    status = "❌ REJECT (miscalibrated)" if rej else "✅ PASS (well-calibrated)"
    print(f"  {label}: p={pv:.4f} → {status}")
print()
print(f"Saved to: {save_path.resolve()}")
print(f"  - lc2st_nf.pkl (trained model)")
print(f"  - lc2st_Tdist.pdf")
print(f"  - lc2st_pp_plot.pdf")
print(f"  - lc2st_pairgrid.pdf")